# Aaru Practice Take-Home — Reference Solution

**Only open this after you've attempted `aaru_practice_takehome.ipynb` yourself, ideally timed.** This is one valid solution, not the only correct one — graders care more about your reasoning and documentation than matching this exactly.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

panel = pd.read_csv('panel_data.csv')
survey = pd.read_csv('survey_data.csv')
print("PANEL:", panel.shape)
print("SURVEY:", survey.shape)


In [ ]:
def quick_eda(df, name):
    print(f"=== {name} ===")
    print(df.dtypes)
    print("\nNulls:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\n", df.head(8))
    print()

quick_eda(panel, "PANEL")
quick_eda(survey, "SURVEY")


In [ ]:
quality_log = []

def log_issue(field, issue, severity, fix):
    quality_log.append({'field': field, 'issue': issue, 'severity': severity, 'recommended_fix': fix})

log_issue('panel_join_date / response_date',
          'Mixed date formats across sources (ISO YYYY-MM-DD in panel, MM/DD/YYYY in survey)',
          'Medium', 'Parse both explicitly with known format strings before any date arithmetic')

log_issue('panelist_name / respondent_full_name',
          'Inconsistent name format across sources ("Last, First" vs "First Last"), plus casing variants and abbreviated first initials',
          'High', 'Normalize both to (first, last) tuples with consistent title-casing before any entity matching')

log_issue('est_annual_income_usd / household_income_band',
          'Income represented as a continuous numeric estimate in panel vs. a categorical band in survey',
          'Medium', 'Bucket the panel numeric estimate into the survey\'s band schema using one shared bucketing function')

log_issue('panelist_zip',
          'Approximately 5% of panel records missing zip code',
          'Low', 'Flag as missing rather than imputing; do not use zip as a hard join key given the gap')

log_issue('panelist_name (duplicates)',
          'Approximately 12% of individuals appear twice in panel data, likely re-enrollment, with slightly different age/income values between the two records',
          'High', 'Deduplicate by resolved entity key, keeping most recent or averaging numeric fields, before downstream use')

log_issue('population overlap',
          'Not every panelist appears in survey data and a handful of survey respondents do not appear in panel data at all',
          'Medium', 'Treat as a left/right partial overlap, not a 1:1 join; explicitly report unmatched counts on both sides')

print(f"{len(quality_log)} issues logged")


In [ ]:
def split_panel_name(name):
    name = name.strip()
    last, first = [p.strip() for p in name.split(',', 1)]
    return first.title(), last.title()

panel[['first_clean', 'last_clean']] = panel['panelist_name'].apply(
    lambda x: pd.Series(split_panel_name(x))
)

def split_survey_name(name):
    parts = name.strip().split(' ', 1)
    return parts[0].title(), parts[1].title()

survey[['first_clean', 'last_clean']] = survey['respondent_full_name'].apply(
    lambda x: pd.Series(split_survey_name(x))
)

# Parse dates explicitly
panel['join_date_parsed'] = pd.to_datetime(panel['panel_join_date'], format='%Y-%m-%d')
survey['response_date_parsed'] = pd.to_datetime(survey['response_date'], format='%m/%d/%Y')

# Zip as nullable string, not float
panel['panelist_zip'] = panel['panelist_zip'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else np.nan
)

# Shared income bucketing function
def bucket_income(x):
    if x < 35000: return '<35k'
    elif x < 65000: return '35-65k'
    elif x < 100000: return '65-100k'
    elif x < 150000: return '100-150k'
    else: return '150k+'

panel['income_band_derived'] = panel['est_annual_income_usd'].apply(bucket_income)

print(panel[['panelist_name','first_clean','last_clean','income_band_derived']].head())
print(survey[['respondent_full_name','first_clean','last_clean']].head())


In [ ]:
# Deduplicate panel BEFORE entity resolution against survey, since duplicates
# are within-source, not cross-source.
panel['block_key'] = panel['last_clean'].str.lower() + '_' + panel['first_clean'].str.lower()
dup_counts = panel['block_key'].value_counts()
dupes = dup_counts[dup_counts > 1]
print(f"{len(dupes)} panelists appear more than once in panel data")

# Keep the most recent enrollment per duplicated panelist
panel_dedup = (
    panel.sort_values('join_date_parsed')
    .drop_duplicates(subset='block_key', keep='last')
    .reset_index(drop=True)
)
print(f"Panel rows before dedup: {len(panel)}, after: {len(panel_dedup)}")


In [ ]:
# Entity resolution: block by last name, match on first name within block
survey['block_key'] = survey['last_clean'].str.lower() + '_' + survey['first_clean'].str.lower()

panel_keys = set(panel_dedup['block_key'])
survey_keys = set(survey['block_key'])

matched_keys = panel_keys & survey_keys
panel_only = panel_keys - survey_keys
survey_only = survey_keys - panel_keys

print(f"Matched (in both sources): {len(matched_keys)}")
print(f"Panel-only (no survey response): {len(panel_only)}")
print(f"Survey-only (not in panel at all): {len(survey_only)}")
print(f"Match rate vs. panel: {len(matched_keys)/len(panel_keys):.1%}")
print(f"Match rate vs. survey: {len(matched_keys)/len(survey_keys):.1%}")


In [ ]:
# Build final integrated dataset: inner join on matched entities, with confidence tagging
merged = panel_dedup.merge(
    survey, on='block_key', how='inner', suffixes=('_panel', '_survey')
)

# Confidence: flag where panel age and survey age bracket disagree
def age_consistent(age, bracket):
    ranges = {'18-29': (18,29), '30-44': (30,44), '45-59': (45,59), '60+': (60,120)}
    lo, hi = ranges[bracket]
    return lo <= age <= hi

merged['match_confidence'] = merged.apply(
    lambda r: 'high' if age_consistent(r['panelist_age'], r['age_bracket']) else 'review', axis=1
)

print(merged['match_confidence'].value_counts())
print(f"\nFinal integrated dataset: {len(merged)} rows, {merged.shape[1]} columns")
merged[['first_clean_panel','last_clean_panel','panelist_age','age_bracket','match_confidence']].head(10)


In [ ]:
print(pd.DataFrame(quality_log))
print()
print(f"SUMMARY")
print(f"  Panel (deduplicated): {len(panel_dedup)} unique panelists")
print(f"  Survey: {len(survey)} respondents")
print(f"  Successfully matched across both sources: {len(matched_keys)} ({len(matched_keys)/len(panel_keys):.1%} of panel)")
print(f"  Panel-only (no survey signal): {len(panel_only)}")
print(f"  Survey-only (not in panel): {len(survey_only)}")
print()
print("Next steps with more time/access:")
print("  - Confirm with the panel vendor whether duplicate enrollments should be averaged or")
print("    most-recent-wins; I assumed most-recent, but this is a business-logic question, not")
print("    purely a technical one.")
print("  - At real scale, name-based blocking alone would miss legitimate variant spellings;")
print("    I'd add a fuzzy-match pass (Jaro-Winkler or similar) on the unmatched residual sets")
print("    before accepting them as true non-matches.")
print("  - Would want a stable cross-source person ID from the source systems if one exists,")
print("    rather than reconstructing identity from name+demographics alone -- much more reliable")
print("    at scale and avoids false merges between different people who share a name.")
